In [2]:
import pandas as pd

# Data folder
data_folder = "/Users/vuongdai/GitHub/bm/detrend_data/weekly/"
data_file = "ts_weekly_values_standardize.csv"
data_file_time = "ts_weekly_datetimes.csv"

output_data_folder = "/Users/vuongdai/GitHub/bm/detrend_data/monthly/"
output_data_file = "ts_monthly_values_standardize.csv"
output_time_file = "ts_monthly_datetimes.csv"

# Read sensor names (header)
sensor_names = pd.read_csv(data_folder + data_file, nrows=0, delimiter=",").columns.tolist()

# Read data and timestamps
df = pd.read_csv(data_folder + data_file, skiprows=1, delimiter=",", header=None)
df_time = pd.read_csv(data_folder + data_file_time, skiprows=1, delimiter=",", header=None)

# Resample each column independently using its own timeline
monthly_data = {}
monthly_times = {}

for i, col in enumerate(sensor_names):
    # Build a small series with this column's own datetime index
    times = pd.to_datetime(df_time.iloc[:, i].dropna())
    values = df.iloc[:len(times), i]

    s = pd.Series(values.values, index=times)

    # Resample to monthly mean
    s_monthly = s.resample("MS").mean()

    monthly_data[col] = s_monthly.values
    monthly_times[col] = s_monthly.index.strftime("%Y-%m-%d")

df_monthly_data = pd.DataFrame(dict([(col, pd.Series(monthly_data[col])) for col in sensor_names]))
df_monthly_times = pd.DataFrame(dict([(col, pd.Series(monthly_times[col])) for col in sensor_names]))

# Write to CSV
df_monthly_data.to_csv(output_data_folder + output_data_file, index=False)
df_monthly_times.to_csv(output_data_folder + output_time_file, index=False)

print(f"Done. Output shape: {df_monthly_data.shape}")
print(f"  Data  → {output_data_file}")
print(f"  Times → {output_time_file}")

Done. Output shape: (303, 101)
  Data  → ts_monthly_values_standardize.csv
  Times → ts_monthly_datetimes.csv
